# Reconstruct metadata

Rebuilds a metadata xlsx file from the conversations JSON, for use when
the metadata file has been lost or corrupted but the conversations JSON
is still intact.

Approach:
1. **Validate** every conversation in the JSON against the Pydantic schema
   for its declared turn count. Report anything malformed and delete it
   from the JSON.
2. **Re-enumerate** the request grid using the same `prompt_dimensions.json`
   and the same flavor seed/count/strategy that was used originally.
   Re-enumeration is deterministic given those inputs.
3. **Match** each enumerated row against a surviving conversation by content
   tuple (template, scenario, representative, caller, benign_context,
   cialdini_emphasis, turn_count_value, flavor, replicate_index). Matched
   rows are marked `success`; the metadata `request_id` is taken from the
   JSON record so the two files stay keyed consistently. Unmatched rows
   stay `pending`.
4. **Write** the new metadata file atomically.

Limitations:
- Per-row retry counts cannot be recovered (the JSON doesn't store them).
  All successful rows are marked `attempts = 1`.
- Conversations whose content tuple doesn't match any enumerated row are
  reported as 'orphaned' and left in the JSON; they are not added to the
  metadata file. This typically means the dimensions or flavor config
  changed between the original run and the reconstruction.

In [ ]:
import json
from collections import Counter
from pathlib import Path
import conversation_generation as cg
from pydantic import ValidationError

## CONFIG

Set the conversations JSON path, the metadata path to write, and the same
enumeration parameters that were used in the original run. Defaults match
the threat-run defaults in `run_generation.ipynb`.

In [ ]:
DIMENSIONS_PATH    = 'prompt_dimensions.json'
CONVERSATIONS_PATH = '../conversations/threat_conversations.json'
METADATA_PATH      = '../conversations/threat_metadata.xlsx'   # will be CREATED / OVERWRITTEN

TEMPLATE_KEY     = 'threat'                 # 'threat' or 'benign'
FLAVOR_COUNT     = 32                       # threat default; use 5 for benign
FLAVOR_STRATEGY  = cg.FLAVOR_DETERMINISTIC  # use cg.FLAVOR_RESAMPLED for benign
FLAVOR_SEED      = 42
REPLICATES       = 1

# Reconstructed-row defaults. These must match the original run for the
# metadata file to be useful for resuming.
MODEL       = 'gpt-5.4'
TEMPERATURE = 1
TOP_P       = 1

# Set False to do a dry run (validates JSON and reports counts, no writes)
WRITE_OUTPUTS = True

## Step 1 - validate conversations JSON

Walks every conversation, validates the parsed dict against
`conversation_model_for(turn_count)`, and collects malformed entries.
Validation also catches records that lack the fields needed to match them
back to a metadata row (selection, flavor, replicate_index).

In [ ]:
store = cg.load_conversation_store(CONVERSATIONS_PATH)
convs = store['conversations']
print(f'Loaded {len(convs)} conversation records from {CONVERSATIONS_PATH}')

REQUIRED_TOP_FIELDS = ['selection', 'flavor', 'replicate_index', 'conversation']
REQUIRED_SELECTION_FIELDS = [
    'prompt_template_key', 'scenario', 'representative', 'caller',
    'benign_context', 'cialdini_emphasis', 'turn_count_value',
]

malformed: list[tuple[str, str]] = []   # (request_id, reason)

for rid, rec in convs.items():
    if not isinstance(rec, dict):
        malformed.append((rid, 'record is not a dict'))
        continue
    missing = [f for f in REQUIRED_TOP_FIELDS if f not in rec]
    if missing:
        malformed.append((rid, f'missing top-level fields: {missing}'))
        continue
    sel = rec.get('selection')
    if not isinstance(sel, dict):
        malformed.append((rid, 'selection is not a dict'))
        continue
    sel_missing = [f for f in REQUIRED_SELECTION_FIELDS if f not in sel]
    if sel_missing:
        malformed.append((rid, f'selection missing fields: {sel_missing}'))
        continue
    try:
        turn_count = int(sel['turn_count_value'])
    except (TypeError, ValueError) as e:
        malformed.append((rid, f'turn_count_value not coercible to int: {e}'))
        continue
    Model = cg.conversation_model_for(turn_count)
    try:
        Model.model_validate(rec['conversation'])
    except ValidationError as e:
        # Keep the message terse for the report
        first = e.errors()[0] if e.errors() else {'msg': str(e)}
        loc = '.'.join(str(p) for p in first.get('loc', []))
        malformed.append((rid, f"schema invalid at {loc}: {first.get('msg', '')}"))

print(f'Malformed records: {len(malformed)}')
for rid, reason in malformed[:10]:
    print(f'  {rid}: {reason}')
if len(malformed) > 10:
    print(f'  ... and {len(malformed) - 10} more')

In [ ]:
# Delete malformed entries from the JSON (atomic write).
if malformed and WRITE_OUTPUTS:
    bad_ids = {rid for rid, _ in malformed}
    cleaned = {rid: rec for rid, rec in convs.items() if rid not in bad_ids}
    cg._atomic_write_text(
        json.dumps({'conversations': cleaned}, indent=2, ensure_ascii=False),
        Path(CONVERSATIONS_PATH),
    )
    print(f'Deleted {len(bad_ids)} malformed records; wrote {len(cleaned)} valid records.')
    convs = cleaned
elif malformed:
    print('WRITE_OUTPUTS=False - leaving malformed records in place.')
    valid_ids = {rid for rid, rec in convs.items() if rid not in {x[0] for x in malformed}}
    convs = {rid: convs[rid] for rid in valid_ids}
else:
    print('All records valid; no JSON cleanup needed.')
print(f'Valid records to use for reconstruction: {len(convs)}')

## Step 2 - re-enumerate requests

Using the same `prompt_dimensions.json` and the same flavor seed/count/strategy,
regenerate the request grid. The combinations that come out are the same as
the original run; the `request_id` UUIDs are different and will be replaced
by JSON IDs for matched rows.

In [ ]:
dimensions = cg.load_dimensions(DIMENSIONS_PATH)
requests = cg.enumerate_requests(
    dimensions,
    template_key    = TEMPLATE_KEY,
    flavor_count    = FLAVOR_COUNT,
    flavor_strategy = FLAVOR_STRATEGY,
    model           = MODEL,
    temperature     = TEMPERATURE,
    top_p           = TOP_P,
    replicates      = REPLICATES,
    flavor_seed     = FLAVOR_SEED,
)
cg.attach_prompts(requests, dimensions)
print(f'Re-enumerated {len(requests)} requests for template {TEMPLATE_KEY!r}.')

## Step 3 - match conversations to enumerated rows

Matching key: the content tuple
`(prompt_template_key, scenario, representative, caller, benign_context,
cialdini_emphasis, turn_count_value, flavor, replicate_index)`.
This uniquely identifies a row given a fixed dimensions+seed config.

In [ ]:
def content_key_from_record(rec: dict) -> tuple:
    sel = rec['selection']
    return (
        sel['prompt_template_key'],
        sel['scenario'],
        sel['representative'],
        sel['caller'],
        sel['benign_context'],
        sel['cialdini_emphasis'],
        int(sel['turn_count_value']),
        rec['flavor'],
        int(rec['replicate_index']),
    )

def content_key_from_row(row: dict) -> tuple:
    return (
        row['prompt_template_key'],
        row['scenario'],
        row['representative'],
        row['caller'],
        row['benign_context'],
        row['cialdini_emphasis'],
        int(row['turn_count_value']),
        row['flavor'],
        int(row['replicate_index']),
    )

# Index conversations by content key. If duplicates exist (shouldn't with
# replicates=1 but possible if a run was started twice), keep the most recent
# by generated_at_utc and report the rest.
by_key: dict[tuple, tuple[str, dict]] = {}   # key -> (request_id, record)
duplicates: list[tuple[str, str]] = []        # list of (kept_id, dropped_id)

for rid, rec in convs.items():
    try:
        key = content_key_from_record(rec)
    except (KeyError, ValueError, TypeError):
        # Should not happen: validated above. Skip defensively.
        continue
    if key in by_key:
        existing_id, existing_rec = by_key[key]
        existing_ts = existing_rec.get('generated_at_utc', '')
        new_ts = rec.get('generated_at_utc', '')
        if new_ts > existing_ts:
            duplicates.append((rid, existing_id))
            by_key[key] = (rid, rec)
        else:
            duplicates.append((existing_id, rid))
    else:
        by_key[key] = (rid, rec)

if duplicates:
    print(f'Found {len(duplicates)} duplicate content-key collisions (kept newer; older listed below)')
    for kept, dropped in duplicates[:5]:
        print(f'  kept {kept[:8]}, also-saw {dropped[:8]}')
    if len(duplicates) > 5:
        print(f'  ... and {len(duplicates) - 5} more')
else:
    print('No duplicate content-key collisions.')

# Match each enumerated row to a JSON record (if any).
matched: dict[str, tuple[str, dict]] = {}    # new_request_id (from row) -> (json_id, record)
unmatched_rows = 0
matched_keys: set[tuple] = set()
for row in requests:
    key = content_key_from_row(row)
    hit = by_key.get(key)
    if hit:
        matched[row['request_id']] = hit
        matched_keys.add(key)
    else:
        unmatched_rows += 1

orphaned_records = [
    (rid, rec) for key, (rid, rec) in by_key.items() if key not in matched_keys
]

print(f'\nMatch summary:')
print(f'  Enumerated rows:           {len(requests)}')
print(f'  Valid JSON records:        {len(convs)}')
print(f'  Matched rows:              {len(matched)} (will be marked success)')
print(f'  Unmatched rows (pending):  {unmatched_rows}')
print(f'  Orphaned records in JSON:  {len(orphaned_records)} (left in place, no matching row)')

if orphaned_records:
    print('\nFirst few orphans:')
    for rid, rec in orphaned_records[:5]:
        sel = rec['selection']
        print(f'  {rid[:8]}: {sel["prompt_template_key"]} / replicate={rec["replicate_index"]} / flavor={rec["flavor"]!r}')

## Step 4 - reconcile JSON request_ids with enumeration

For matched rows, copy the JSON record's `request_id` onto the enumerated
row, and update each row with the data we know from the conversation
(prompts, usage, generation timestamp, extracted flag metrics, success status).

In [ ]:
for row in requests:
    hit = matched.get(row['request_id'])
    if hit is None:
        # leave row in 'pending' state for re-generation later
        continue
    json_id, rec = hit
    # Adopt the JSON's request_id so the metadata + JSON stay consistent.
    row['request_id'] = json_id
    # Pull through prompts (may not have been re-attached identically if
    # any template text drifted; trust the original record).
    if 'system_prompt' in rec:
        row['system_prompt'] = rec['system_prompt']
    if 'user_prompt' in rec:
        row['user_prompt'] = rec['user_prompt']
    # Usage
    usage = rec.get('usage') or {}
    row['input_tokens']  = usage.get('input_tokens', 0) or 0
    row['output_tokens'] = usage.get('output_tokens', 0) or 0
    row['total_tokens']  = usage.get('total_tokens', 0) or 0
    # Flag metrics (re-derived from the conversation)
    metrics = cg.extract_flag_metrics(rec['conversation'], int(row['turn_count_value']))
    row.update(metrics)
    # Status fields
    row['status']  = 'success'
    row['attempts'] = 1   # original attempt count not recoverable
    row['last_error'] = ''
    row['generated_at_utc'] = rec.get('generated_at_utc', '')

status_counter = Counter(r['status'] for r in requests)
print(f'Reconciled row statuses: {dict(status_counter)}')

## Step 5 - write the new metadata file

Atomic write. If `WRITE_OUTPUTS` is False, this cell prints what would be
written but doesn't touch the disk.

In [ ]:
if WRITE_OUTPUTS:
    cg.write_metadata_xlsx(requests, METADATA_PATH)
    print(f'Wrote {METADATA_PATH} ({len(requests)} rows)')
    rows_back = cg.read_metadata_xlsx(METADATA_PATH)
    print(f'Verification read-back: {len(rows_back)} rows  statuses {dict(Counter(r["status"] for r in rows_back))}')
else:
    print(f'WRITE_OUTPUTS=False - would have written {len(requests)} rows to {METADATA_PATH}.')

## Step 6 - sanity check

Pick one matched row and confirm the metric columns reflect the
conversation in the JSON.

In [ ]:
if WRITE_OUTPUTS:
    rows_back = cg.read_metadata_xlsx(METADATA_PATH)
    succ = [r for r in rows_back if r['status'] == 'success']
    if succ:
        r = succ[0]
        rec = convs[r['request_id']]
        print(f'request_id: {r["request_id"]}')
        print(f'flavor:     {r["flavor"]!r}')
        print(f'turn_count: {r["turn_count_value"]}')
        print(f'tokens:     in={r["input_tokens"]}  out={r["output_tokens"]}  total={r["total_tokens"]}')
        print()
        # Spot-check one flag metric against the source conversation
        recomputed = cg.extract_flag_metrics(rec['conversation'], int(r['turn_count_value']))
        for f in cg.CIALDINI_FLAGS + cg.IMPROPER_FLAGS:
            assert r[f'{f}_sum'] == recomputed[f'{f}_sum'], f'mismatch on {f}_sum'
            assert r[f'{f}_by_turn'] == recomputed[f'{f}_by_turn'], f'mismatch on {f}_by_turn'
        print('All metric columns match recomputed values from JSON.')